In [ ]:
import pandas as pd

df = pd.read_csv("../data/NGX_Stocks_2000-2025.csv")
display(df.head())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date')
display(df.head())

In [ ]:
# Calculate 20-day SMA
df['SMA_20'] = df['Price'].rolling(window=20).mean()

# Calculate 50-day SMA
df['SMA_50'] = df['Price'].rolling(window=50).mean()

# Calculate Price Change for RSI
delta = df['Price'].diff()
gain = delta.mask(delta < 0, 0)
loss = -delta.mask(delta > 0, 0)

# Calculate Average Gain and Average Loss
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()

# Calculate RS and RSI
rs = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# Calculate MACD
ema_12 = df['Price'].ewm(span=12, adjust=False).mean()
ema_26 = df['Price'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema_12 - ema_26

# Calculate MACD Signal Line
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

display(df.head())

In [ ]:
print("Columns with missing values before handling:")
print(df.isnull().sum())

df.fillna(method='ffill', inplace=True)

print("\nColumns with missing values after handling:")
print(df.isnull().sum())

display(df.head())

In [ ]:
df.dropna(subset=['SMA_20', 'SMA_50', 'RSI_14'], inplace=True)

print("Columns with missing values after dropping initial rows:")
print(df.isnull().sum())

display(df.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cols_to_scale = [col for col in numerical_cols if col not in ['Symbol', 'Sector', 'index']]

scaler = MinMaxScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

display(df.head())

In [ ]:
train_size = int(len(df) * 0.7)
val_size = int(len(df) * 0.15)
test_size = len(df) - train_size - val_size

train_df = df[:train_size]
val_df = df[train_size:train_size + val_size]
test_df = df[train_size + val_size:]

print("Shape of training data:", train_df.shape)
print("Shape of validation data:", val_df.shape)
print("Shape of testing data:", test_df.shape)

In [ ]:
!pip install stable-baselines3 gymnasium

In [ ]:
for col in df.columns:
    if df[col].dtype == 'object':
        print(f"Column '{col}' has object dtype.")
        unique_values = df[col].unique()
        print(f"Unique values in '{col}': {unique_values[:20]}")

In [ ]:
df['Change %'] = df['Change %'].astype(str).str.replace('%', '').str.replace(',', '').astype(float)
display(df.head())

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO

class TradingEnvironment(gym.Env):
    """A custom trading environment for OpenAI Gym."""

    def __init__(self, dataframe, transaction_cost_pct=0.0, slippage_pct=0.0005):
        super().__init__()
        self.df = dataframe
        self.current_step = 0
        self.initial_balance = 10000  # Starting balance
        self.balance = self.initial_balance
        self.shares_held = 0
        self.net_worth = self.initial_balance
        self.max_net_worth = self.initial_balance
        self.window_size = 14  # Example window size for observation
        self.transaction_cost_pct = transaction_cost_pct # Store transaction cost
        self.slippage_pct = slippage_pct # Store slippage percentage

        # Action space: 0: Hold, 1: Buy, 2: Sell
        self.action_space = spaces.Discrete(3)

        # Observation space: Price, Volume, and calculated indicators within a window

        numerical_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        self._features = [col for col in numerical_cols if col not in ['index']] # Exclude 'index'

        # The observation space will be a flattened array of the windowed features
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(len(self._features) * self.window_size,), dtype=np.float32
        )

    def _next_observation(self):
        # the observation window
        start = self.current_step - self.window_size + 1
        end = self.current_step + 1
        # To Ensure only numerical features are selected for the observation
        obs_df = self.df.iloc[start:end][self._features]
        return obs_df.values.flatten()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size - 1  # To Start after the first window
        self.balance = self.initial_balance
        self.shares_held = 0
        self.net_worth = self.initial_balance
        self.max_net_worth = self.initial_balance

        observation = self._next_observation()
        info = {}
        return observation, info

    def step(self, action):
        self.current_step += 1
        if self.current_step >= len(self.df):
            done = True
            self.current_step = len(self.df) - 1
        else:
            done = False

        current_price = self.df.iloc[self.current_step]['Price']
        executed_price = current_price

        # Apply slippage and execute trading action
        if action == 1:  # Buy
            # Calculate slippage for buy (increase in price)
            slippage_amount = current_price * self.slippage_pct
            executed_price = current_price + slippage_amount

            # Buy as many shares as possible with the current balance
            if executed_price > 0: # Avoid division by zero
                buy_shares = self.balance // executed_price
                if buy_shares > 0:
                    cost = buy_shares * executed_price
                    transaction_cost = cost * self.transaction_cost_pct
                    if self.balance >= cost + transaction_cost:
                        self.balance -= (cost + transaction_cost)
                        self.shares_held += buy_shares

        elif action == 2:  # Sell
            # Calculate slippage for sell (decrease in price)
            slippage_amount = current_price * self.slippage_pct
            executed_price = current_price - slippage_amount

            # Sell all shares held
            if self.shares_held > 0 and executed_price > 0: # To Avoid negative executed price and division by zero
                sale_value = self.shares_held * executed_price
                transaction_cost = sale_value * self.transaction_cost_pct
                self.balance += (sale_value - transaction_cost)
                self.shares_held = 0

        # Calculate net worth
        prev_net_worth = self.net_worth
        self.net_worth = self.balance + self.shares_held * current_price # Net worth based on current price, not executed price
        self.max_net_worth = max(self.max_net_worth, self.net_worth)

        # Calculate reward (percentage change in net worth)
        reward = (self.net_worth - prev_net_worth) / prev_net_worth if prev_net_worth > 0 else 0

        observation = self._next_observation()
        info = {
            'balance': self.balance,
            'shares_held': self.shares_held,
            'net_worth': self.net_worth,
            'max_net_worth': self.max_net_worth,
            'current_price': current_price,
            'executed_price': executed_price # Added executed price to info
        }

        # Additional termination condition: run out of data
        if self.current_step >= len(self.df) - 1:
            done = True

        return observation, reward, done, False, info

In [ ]:
# Create the training Environment
train_env = TradingEnvironment(train_df, transaction_cost_pct=0.0, slippage_pct=0.0005) # Added slippage_pct

# Import PPO
from stable_baselines3 import PPO

# Initialize and train the PPO agent
model = PPO("MlpPolicy", train_env, verbose=1, learning_rate=0.0001, n_steps=2048) # Adjusted hyperparameters

# Training the agent
model.learn(total_timesteps=50000) # Increased total_timesteps

# Save the model
model_save_path = "./ppo_trading_agent"
model.save(model_save_path)

print(f"Model saved to {model_save_path}")

In [ ]:
# Evaluation
import numpy as np
import pandas as pd

# Creating a testing environment
test_env = TradingEnvironment(test_df, transaction_cost_pct=0.0, slippage_pct=0.0005) # Added slippage_pct

# Reset the testing environment
obs, info = test_env.reset()

# Initialize variables to track performance
cumulative_return = 0
net_worth_history = [test_env.initial_balance]

# Loop through the testing environment steps
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True) # Use deterministic action for evaluation
    obs, reward, done, truncated, info = test_env.step(action)
    net_worth_history.append(info['net_worth'])

# Calculate cumulative return
final_net_worth = net_worth_history[-1]
cumulative_return = (final_net_worth - test_env.initial_balance) / test_env.initial_balance

# Calculate daily returns
net_worth_series = pd.Series(net_worth_history)
daily_returns = net_worth_series.pct_change().dropna()

# Calculate Sharpe ratio (assuming risk-free rate of 0)
# small epsilon added to the standard deviation to avoid division by zero
sharpe_ratio = daily_returns.mean() / (daily_returns.std() + 1e-9) * np.sqrt(252) # Assuming 252 trading days in a year

# Print the results
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot the net worth history
plt.figure(figsize=(12, 6))
plt.plot(net_worth_history)
plt.title('Net Worth Over Time')
plt.xlabel('Trading Days (since start of test data)')
plt.ylabel('Net Worth')
plt.grid(True)
plt.show()